# Building the Knowledge Model

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

## Loading the Clean Dataset

In [2]:
print('-----------------------------------------')
print('LOADING DATASET A FILE DOWNLOAD DATASET')
data_file = 'data/Op1_CapDL_NoSteps.csv'
data = pd.read_csv(data_file, sep=",", decimal='.', low_memory = False)
data = data.drop(columns=['Unnamed: 0'])
print('Data Size:(%d, %d)'%(data.shape[0], data.shape[1]))
print('-----------------------------------------')
target_kpi = 'transfer.datarate'
data

-----------------------------------------
LOADING DATASET A FILE DOWNLOAD DATASET
Data Size:(1001, 134)
-----------------------------------------


,resets.sent.a2b,ack.pkts.sent.a2b,pure.acks.sent.b2a,sack.pkts.sent.b2a,dsack.pkts.sent.a2b,actual.data.bytes.b2a,rexmt.data.bytes.a2b,rexmt.data.bytes.b2a,outoforder.pkts.b2a,pushed.data.pkts.b2a,...,abs.instanttp.75.,abs.instanttp.max,abs.burst.sum,abs.rtoevents.sum,abs.theoricalmaxtp.25.,abs.theoricalmaxtp.50.,abs.theoricalmaxtp.avg,abs.theoricalmaxtp.75.,abs.theoricalmaxtp.max,transfer.datarate
0,100.0,24629.0,8.0,0.0,43.0,405175766.0,0.0,439348.0,17.0,14030.0,...,514.41896,797.85984,14.0,3.0,504.884889,608.801561,3.983938e+09,731.608632,3.131014e+13,448375.109
1,198.0,15636.0,42.0,0.0,22.0,133444830.0,0.0,72488.0,3190.0,6355.0,...,327.40704,489.87312,191.0,8.0,121.466483,309.086095,3.452638e+02,483.573895,6.846370e+04,131054.310
2,5.0,25907.0,6.0,0.0,40.0,457404418.0,0.0,440036.0,951.0,13224.0,...,604.88512,898.44344,111.0,4.0,582.045037,740.081545,4.703781e+10,1068.091429,7.084634e+13,491072.262
3,126.0,26205.0,5.0,0.0,0.0,464522131.0,0.0,0.0,249.0,13853.0,...,611.17992,895.45944,32.0,0.0,582.787765,747.875139,5.891454e+09,968.802286,1.341955e+14,515530.559
4,146.0,27219.0,5.0,0.0,2.0,487527222.0,0.0,6940.0,1288.0,13826.0,...,600.17888,884.71696,189.0,0.0,667.284364,863.544471,6.077600e+09,1239.483809,5.493232e+13,534287.880
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,113.0,25235.0,8.0,0.0,0.0,489252066.0,0.0,0.0,1472.0,13635.0,...,622.36856,835.97208,134.0,0.0,684.201143,837.093600,3.259872e+09,1332.840949,7.786691e+13,531207.713
997,18.0,13620.0,46.0,0.0,62.0,178992008.0,0.0,499916.0,331.0,7572.0,...,297.11856,406.61320,30.0,126.0,197.677920,257.693275,4.374948e+09,337.645500,2.416830e+13,200766.911
998,175.0,21683.0,72.0,0.0,329.0,366177754.0,0.0,2819015.0,146.0,11669.0,...,559.98144,712.17808,59.0,25.0,497.796398,650.059130,1.583619e+10,858.313176,5.087490e+13,392487.966
999,239.0,22295.0,16.0,0.0,68.0,309297311.0,0.0,278111.0,61.0,12047.0,...,442.76792,708.73160,16.0,19.0,383.610240,453.332068,4.499866e+02,507.976497,4.856150e+03,336845.238


## Detection and Removal of Features Correlated with the KPI

In [ ]:
# Compute the matrix of correlation factors and extract the vector related to the target KPI 
corr_matrix = 1 - data.corr(method='pearson').abs()
corr_vector = corr_matrix[target_kpi].to_numpy().reshape(-1,1)

eval_window = 40
max_num_clusters = np.min([eval_window + 1, len(corr_vector)])

iterations = 50
bic_ite    = np.zeros((iterations, max_num_clusters - 1))
aic_ite    = np.zeros((iterations, max_num_clusters - 1))
rand_stt   = np.random.randint(2**16, size=(iterations, max_num_clusters - 1)) 

for ii in range(iterations):
    for jj in range(1, max_num_clusters):
        gmm = mixture.GaussianMixture(n_components = jj,
                                      random_state = rand_stt[ii,jj-1])
        gmm.fit(corr_vector)
        bic_ite[ii,jj-1] = gmm.bic(corr_vector)
        aic_ite[ii,jj-1] = gmm.aic(corr_vector)
        
num_opt_clusters_bic = np.argmin(np.mean(bic_ite, axis=0)) + 1
num_opt_clusters_aic = np.argmin(np.mean(aic_ite, axis=0)) + 1

print(num_opt_clusters_bic, num_opt_clusters_aic)